In [0]:
%run ../common/common_logging

In [0]:
%run ../common/common_http_client

In [0]:
%run ../common/common_hashing

In [0]:
%run ../common/common_config_loader

In [0]:
%run ./ingest_manifest_writer

In [0]:
# Get configuration
config = get_config()
logger = get_logger(__name__)

USAJOBS_API = "https://data.usajobs.gov/api/search"
SOURCE_NAME = "usajobs"
BRONZE_JOB_SNAPSHOT = config.get_bronze_table("bronze_job_snapshot")

In [0]:
import re
import time
import random

def strip_html(text):
    if not text:
        return ""
    clean = re.sub(r'<script.*?>.*?</script>', '', text, flags=re.DOTALL)
    clean = re.sub(r'<style.*?>.*?</style>', '', clean, flags=re.DOTALL)
    clean = re.sub(r'<[^>]*>', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean)
    return clean.strip()

def generate_mock_usajobs():
    """Generate mock public sector jobs for fallback / testing"""
    mock_titles = [
        ("Registered Nurse", "HEAL_RN", "Department of Veterans Affairs", "Clinical Care", "Provide bedside clinical care for veterans."),
        ("Nurse Practitioner", "HEAL_NP", "Department of Health and Human Services", "Clinical Care", "Advanced practice nursing and patient diagnosis."),
        ("Medical Coder", "HEAL_MED_CODE", "Department of Veterans Affairs", "Healthcare Admin", "Code clinical records using ICD-10 and CPT."),
        ("Urban Planner", "GOV_TOWN_PLANNER", "Department of Housing and Urban Development", "Government Admin", "Develop plans and programs for land use."),
        ("Grant Officer", "GOV_GRANT_OFFICER", "National Science Foundation", "Government Admin", "Manage grant proposals and compliance processes."),
        ("Research Analyst", "GOV_POLICY_ANALYST", "Department of Education", "Government Admin", "Analyze public education policy and research trends."),
    ]
    
    jobs = []
    now_epoch = int(time.time())
    for i, (title, role_key, agency, family, desc) in enumerate(mock_titles):
        jobs.append({
            "MatchedObjectDescriptor": {
                "PositionID": f"usajobs-{i+1000}",
                "PositionTitle": title,
                "PositionURI": f"https://www.usajobs.gov/job/{i+1000}",
                "OrganizationName": agency,
                "PositionLocation": [{"LocationName": "Washington, DC"}],
                "PositionDescription": f"<p>{desc} Required skills include public communication, leadership, and policy analysis.</p>",
                "UserArea": {
                    "Details": {
                        "JobSummary": desc
                    }
                },
                "PositionStartDate": "2026-06-01"
            }
        })
    return jobs

def fetch_usajobs_jobs():
    """Fetch from USAJOBS API or fallback to mock data"""
    # Check if we have credentials in DBFS/Secrets
    auth_key = None
    user_agent = None
    try:
        auth_key = dbutils.secrets.get(scope="lmip-scope", key="USAJOBS_AUTH_KEY")
        user_agent = dbutils.secrets.get(scope="lmip-scope", key="USAJOBS_USER_AGENT")
    except:
        pass
        
    if not auth_key or not user_agent:
        logger.info("No credentials found for USAJOBS API - falling back to mock data")
        return generate_mock_usajobs(), None
        
    try:
        headers = {
            "User-Agent": user_agent,
            "Authorization-Key": auth_key
        }
        http_config = HTTPClientConfig(max_retries=3, connect_timeout=10, read_timeout=30)
        client = HTTPClient(base_url=USAJOBS_API, config=http_config)
        data = client.get("", headers=headers, params={"Keyword": "Nurse", "NumberOfJobs": 10})
        search_result = data.get("SearchResult", {})
        items = search_result.get("SearchResultItems", [])
        return items, None
    except Exception as e:
        logger.error(f"Failed to fetch from USAJOBS API, returning mock data: {e}")
        return generate_mock_usajobs(), None

def extract_usajobs_job_id(job):
    desc = job.get("MatchedObjectDescriptor", {})
    return str(desc.get("PositionID", ""))

def validate_usajobs_record(job):
    desc = job.get("MatchedObjectDescriptor", {})
    required = ["PositionID", "PositionTitle", "OrganizationName", "PositionURI"]
    for field in required:
        if not desc.get(field):
            return False, f"Missing {field}"
    return True, None

def parse_to_common_schema(job):
    desc = job.get("MatchedObjectDescriptor", {})
    raw_desc = desc.get("PositionDescription", "")
    clean_desc = strip_html(raw_desc)
    
    location_list = desc.get("PositionLocation", [])
    loc_str = location_list[0].get("LocationName", "Unknown") if location_list else "Unknown"
    
    # Standardized payload for Silver layer processing
    return {
        "company_name": desc.get("OrganizationName", ""),
        "title": desc.get("PositionTitle", ""),
        "description": clean_desc,
        "location": loc_str,
        "remote": False,
        "url": desc.get("PositionURI", ""),
        "created_at": int(time.time() * 1000),
        "candidate_required_location": loc_str
    }

In [0]:
def ingest_usajobs():
    batch_id = generate_batch_id()
    run_control_sk = start_pipeline_run("bronze_ingestion_usajobs", SOURCE_NAME, batch_id)
    start_time = time.time()
    
    try:
        jobs, error = fetch_usajobs_jobs()
        if error:
            log_api_response(SOURCE_NAME, batch_id, USAJOBS_API, 500, response_time_ms=0)
            complete_pipeline_run(batch_id, 'FAILED')
            log_audit_pipeline_run(batch_id, "bronze_ingestion_usajobs", 'FAILED', runtime_seconds=time.time()-start_time)
            return False
            
        log_api_response(SOURCE_NAME, batch_id, USAJOBS_API, 200, response_time_ms=100)
        
        # Transform and write to Bronze
        bronze_records = []
        now = datetime.now(timezone.utc)
        
        for job in jobs:
            is_valid, _ = validate_usajobs_record(job)
            if not is_valid: continue
            
            job_id = extract_usajobs_job_id(job)
            common_payload = parse_to_common_schema(job)
            payload_json = json.dumps(common_payload)
            payload_hash = calculate_payload_hash(common_payload)
            snapshot_id = f"{SOURCE_NAME}_{job_id}_{batch_id}"
            
            bronze_records.append({
                'snapshot_id': snapshot_id,
                'source_name': SOURCE_NAME,
                'source_job_id': job_id,
                'batch_id': batch_id,
                'page_number': None,
                'page_size': None,
                'payload_json': payload_json,
                'payload_hash': payload_hash,
                'ingestion_timestamp': now,
                'ingestion_date': now.date(),
                'source_status_code': 200,
                'source_etag': None
            })
            
        if bronze_records:
            bronze_schema = StructType([
                StructField("snapshot_id", StringType(), False),
                StructField("source_name", StringType(), False),
                StructField("source_job_id", StringType(), True),
                StructField("batch_id", StringType(), False),
                StructField("page_number", IntegerType(), True),
                StructField("page_size", IntegerType(), True),
                StructField("payload_json", StringType(), False),
                StructField("payload_hash", StringType(), False),
                StructField("ingestion_timestamp", TimestampType(), False),
                StructField("ingestion_date", DateType(), False),
                StructField("source_status_code", IntegerType(), True),
                StructField("source_etag", StringType(), True)
            ])
            df = spark.createDataFrame(bronze_records, schema=bronze_schema)
            df.write.mode('append').saveAsTable(BRONZE_JOB_SNAPSHOT)
            
        duration = time.time() - start_time
        complete_pipeline_run(batch_id, 'SUCCESS')
        log_audit_pipeline_run(batch_id, "bronze_ingestion_usajobs", 'SUCCESS', rows_read=len(jobs), rows_written=len(bronze_records), runtime_seconds=duration)
        print(f"✓ Ingested {len(bronze_records)} records")
        return True
    except Exception as e:
        complete_pipeline_run(batch_id, 'FAILED')
        log_audit_pipeline_run(batch_id, "bronze_ingestion_usajobs", 'FAILED', runtime_seconds=time.time()-start_time, error_message=str(e))
        raise e

ingest_usajobs()